## **LayerSeg - Layered Structure Segmentation Tool**

**LayerSeg** is a UNet-based segmentation tool for binarizing cross-section images of layered materials (e.g., SEM micrographs of graphene oxide membranes). The model was trained on synthetic data and is designed to highlight **horizontal lamellar boundaries** while suppressing vertical features.

Because the model was trained on synthetic data, there is a domain gap with real images. The tool generates masks at **multiple threshold values** so that an expert can visually compare them with the original and select the optimal binarization.

The resulting binary masks can be used for quantitative analysis of angular distributions and orientational order parameters (see [Orientational Ordering of Graphene Oxide Membranes by a Spin Probe Technique and SEM Image Analysis, J. Phys. Chem. C, 2024](https://pubs.acs.org/doi/abs/10.1021/acs.jpcc.3c07127)).

### **User Guide**

**How to use this notebook:**

1. **Section 1**: Run cells 1.1-1.3 to set up the environment (downloads the project and installs dependencies)
2. **Section 2**: Configure parameters (report name, scale factors, blur, thresholds, color of interest)
3. **Section 3**: Upload your document images
4. **Section 4**: Run the binarization pipeline
5. **Section 5**: Download the generated reports (ZIP archives with masks and visualizations)

In [ ]:
#@title **1.1 Importing necessary libraries**
import os
from datetime import datetime
import yaml
import numpy as np
from google.colab import files
from pathlib import Path
time_now = datetime.now().strftime("%d_%H_%M")

print("Libraries imported successfully.")

In [ ]:
#@title **1.2 Downloading and preparing the program**

!git clone https://github.com/NMar33/layerseg
WORKDIR = "/content/layerseg"

CONFIG_PATH = "configs/config_colab.yaml"
CLI_PATH = "src/binarizer_cli.py"
ORIGINAL_IMG_PATH = WORKDIR + "/data/original_imgs"
IMGS_IN_ROW = 3

import os
os.chdir(WORKDIR)

!pip install -r "requirements-colab.txt"
!python download_model.py

import yaml
with open(CONFIG_PATH, "r") as input_stream:
  configs = yaml.safe_load(input_stream)

print("Setup complete.")

In [ ]:
#@title **1.3 Verifying program download**
if Path(CLI_PATH).exists():
  print("Program has been successfully downloaded.")
else:
  print("Error: Program download failed.")

In [ ]:
#@title **2.1 Setting up report parameters**
#@markdown Please enter the name for the report
report_name = "test_1" #@param {type:"string"}
report_name = str(report_name)

print(f"Report parameters set successfully. Report name: '{report_name}'.")

In [ ]:
#@title **2.2 Setting up Preprocessing Parameters**
#@markdown `scale_factors`: The scaling factors to apply to the images. If the factor is > 1, the image will be upscaled, and if it's < 1, the image will be downscaled. Typically, scale factors > 1 are suitable for small images.

#@markdown `gaussian_blur`: If set to 'True', Gaussian blur will be applied to the images. This can be useful for reducing noise in images with fine details or artifacts.

#@markdown `gaussian_blur_kernel_size`: The size of the Gaussian blur kernel. Larger kernel sizes produce a more significant blurring effect.

scale_factors = "1" #@param {type:"string"}

gaussian_blur = "True" #@param ["True", "False"]
gaussian_blur_kernel_size = 7 #@param {type:"slider", min:3, max:15, step:2}

scale_factors = list(map(float, scale_factors.replace(" ", "").split(",")))
gaussian_blur = (gaussian_blur == "True")
gaussian_blur_kernel_size = int(gaussian_blur_kernel_size)

print(f"Preprocessing parameters set successfully:\n- Scale factors: {scale_factors}\n- Gaussian blur: {gaussian_blur}\n- Gaussian blur kernel size: {gaussian_blur_kernel_size}")

In [ ]:
#@title **2.3 Setting up Binarization Parameters**
#@markdown `model_name`: The name of the pre-trained model to be used for image segmentation.

#@markdown `binarizer_thresholds_min` and `binarizer_thresholds_max` define the range of binarization thresholds to be used in the analysis. The binarization thresholds will be evenly spaced between the minimum and maximum values, with the number of steps specified by `binarizer_thresholds_steps`.

#@markdown The `color_interest` parameter specifies the color of the area of interest in the image. Select "black" if the lines of interest are black, or "white" if the areas of interest are white.

model_name = "unet_220805.pth" #@param ["unet_220805.pth"]

binarizer_thresholds_min = 0.63 #@param {type:"slider", min:0.5, max:0.85, step:0.01}
binarizer_thresholds_max = 0.77 #@param {type:"slider", min:0.5, max:0.99, step:0.01}
binarizer_thresholds_steps = 15 #@param {type:"slider", min:3, max:30, step:3}
color_interest = "black" #@param ["black", "white"]

binarizer_thresholds = np.linspace(binarizer_thresholds_min, binarizer_thresholds_max, binarizer_thresholds_steps).tolist()
color_interest = str(color_interest)
print(f"Binarization parameters set successfully:\n- Binarizer thresholds range: {binarizer_thresholds_min} to {binarizer_thresholds_max} with {binarizer_thresholds_steps} steps\n- Color of interest: {color_interest}")

In [ ]:
#@title **2.4 Saving Settings**
configs["report_name"] = report_name
configs["scale_factors"] = scale_factors
configs["gaussian_blur"] = gaussian_blur
configs["gaussian_blur_kernel_size"] = gaussian_blur_kernel_size
configs["model_name"] = model_name
configs["binarizer_thresholds"] = binarizer_thresholds
configs["color_interest"] = color_interest

with open(CONFIG_PATH, "w") as f:
  yaml.dump(configs, f)

print("Settings have been successfully saved.")

In [ ]:
#@title **3.1 Uploading Original Images**
#@markdown Upload the original images for binarization.

print("Please upload your original images...")
os.chdir(ORIGINAL_IMG_PATH)
uploaded_files = files.upload()

if len(uploaded_files) > 0:
    print(f"Successfully uploaded {len(uploaded_files)} original images.")
else:
    print("No original images were uploaded.")

os.chdir(WORKDIR)

In [ ]:
#@title **4.1 Execute Image Segmentation**
#@markdown Run the segmentation pipeline on the uploaded images with the configured settings. The tool will generate soft probability masks and binary masks at each threshold value.

!python src/binarizer_cli.py -cfg configs/config_colab.yaml

In [ ]:
#@title **5.1 Archiving the Report**
#@markdown Create zipped archives for the short and full reports.
os.environ["REPORT_NAME"] = f"report_{time_now}.zip"
os.environ["REPORT_SHORT_NAME"] = f"report_short_{time_now}.zip"

# Create zipped archives for the short and full reports
!zip -r $REPORT_SHORT_NAME reports_short
!zip -r $REPORT_NAME reports

In [ ]:
#@title **5.2 Download the Report Short**
#@markdown To successfully download the report, please make sure to allow file downloads in your browser settings.
files.download(f"report_short_{time_now}.zip")

In [ ]:
#@title **5.3 Download the Report Full**
#@markdown To successfully download the report, please make sure to allow file downloads in your browser settings.
files.download(f"report_{time_now}.zip")